# <font color='orange'>**RapidRelief AI — Sprint 2 & 3: Transferencia de Aprendizaje**</font>

**Metodología:** Igual que en clase — MobileNetV2 + `include_top=False` + cabeza personalizada  
**Datasets:** Clothing Small (imágenes reales) + Fashion-MNIST (imágenes sintéticas complementarias)  
**Meta:** `val_accuracy ≥ 0.90`

## 1. Importar el modelo de interés

Igual que en el notebook de clase — MobileNetV2 con `TF_USE_LEGACY_KERAS`.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
%matplotlib inline

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Rutas del proyecto
BASE_DIR     = '/content/drive/MyDrive/RapidReliefAI'
CLOTH_TRAIN  = BASE_DIR + '/Datasets/clothing_small/train'
CLOTH_VAL    = BASE_DIR + '/Datasets/clothing_small/validation'
CLOTH_TEST   = BASE_DIR + '/Datasets/clothing_small/test'
FMNIST_TRAIN = BASE_DIR + '/Datasets/fashion_mnist/train/fashion-mnist_train.csv'
MODELS_DIR   = BASE_DIR + '/Models'

# Parametros — mismos que en clase
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
N_CLASES   = 10

# Etiquetas finales (orden alfabetico = orden que asigna flow_from_directory)
CLASES = ['dress', 'hat', 'longsleeve', 'outwear', 'pants',
          'shirt', 'shoes', 'shorts', 'skirt', 't-shirt']
print('Configuracion lista')
print(f'Clases ({N_CLASES}):', CLASES)

## 2. Importar la ruta del DATASET

Mismo patrón que en clase: `ImageDataGenerator` + `flow_from_directory` + `preprocessing_function=preprocess_input`.

In [ ]:
# Generadores — misma estructura que en clase
# Augmentation validado en context/referent.md para este dominio de prendas
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    shear_range=10.0,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

print('Probando imagenes')
train_data = train_datagen.flow_from_directory(
    CLOTH_TRAIN,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

val_data = val_datagen.flow_from_directory(
    CLOTH_VAL,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

test_data = val_datagen.flow_from_directory(
    CLOTH_TEST,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=42
)

In [ ]:
# Verificar forma del batch — igual que en clase
imgs, labels = next(train_data)
print(f'Forma imagenes: {imgs.shape}')   # (32, 224, 224, 3)
print(f'Forma etiquetas: {labels.shape}') # (32, 10)
print('Indice de clases:', train_data.class_indices)

In [ ]:
# Visualizar batch — misma funcion que en clase
def plotImages(images_arr):
    fig, axes = plt.subplots(1, 10, figsize=(20, 20))
    axes = axes.flatten()
    for img, ax in zip(images_arr, axes):
        # Revertir preprocess_input para visualizacion
        img_vis = (img + 1.0) / 2.0
        ax.imshow(np.clip(img_vis, 0, 1))
        ax.axis('off')
    plt.tight_layout()
    plt.show()

plotImages(imgs)
print(labels[:10])
# Etiquetas: 0=dress 1=hat 2=longsleeve 3=outwear 4=pants 5=shirt 6=shoes 7=shorts 8=skirt 9=t-shirt

## 3. Cargar el modelo pre-entrenado

Igual que en clase: `include_top=False` para no cargar las capas densas de ImageNet,  
y `trainable = False` para congelar todos los pesos del modelo base.

In [ ]:
# Al cargar un modelo, el argumento include_top se establece en False
# para que las capas de salida densas del modelo no sean cargadas,
# permitiendo agregar y entrenar una nueva capa de salida
pre_trained_model = MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

In [ ]:
# Se debe congelar el modelo base — no se ajustaran los pesos del modelo pre-entrenado
pre_trained_model.trainable = False

### <font color='darkviolet'>**Definir utilizando Modelo Funcional de Keras para el entrenamiento**</font>

In [ ]:
pre_trained_model.output

In [ ]:
pre_trained_model.input

In [ ]:
pre_trained_model.summary()

In [ ]:
# Agregar nuevas capas utilizando la API funcional de KERAS
# Misma estructura que en clase, adaptada a 10 clases
x = Flatten()(pre_trained_model.output)
x = Dense(2048, activation='relu')(x)
x = Dense(512, activation='relu')(x)
x = Dense(16, activation='sigmoid')(x)
predicciones = Dense(N_CLASES, activation='softmax')(x)

In [ ]:
# Crear modelo — API funcional de Keras (igual que en clase)
modelo = Model(inputs=pre_trained_model.input, outputs=predicciones)
modelo.summary()

In [ ]:
# Compilar modelo — igual que en clase
modelo.compile(
    loss='categorical_crossentropy',
    optimizer='sgd',
    metrics=['accuracy']
)

## 4. Entrenar modelo

In [ ]:
# Callback para guardar el mejor modelo (igual que en referent.md)
os.makedirs(MODELS_DIR, exist_ok=True)

callbacks = [
    ModelCheckpoint(
        MODELS_DIR + '/rapidrelief_v1_{epoch:02d}_{val_accuracy:.3f}.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    )
]

# Entrenar modelo — igual que en clase
history = modelo.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)

## 5. Guardar el modelo

In [ ]:
# Crear carpeta si no existe
!mkdir -p {MODELS_DIR}

# Guardar modelo en formato H5 — igual que en clase
modelo.save(MODELS_DIR + '/rapidrelief_mobilenetv2_v1.h5')
print('Modelo guardado en:', MODELS_DIR + '/rapidrelief_mobilenetv2_v1.h5')

## 6. Curvas de entrenamiento

In [ ]:
# Exactitud del modelo — misma grafica que en clase
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'], 'r--')
plt.title('Exactitud del modelo')
plt.ylabel('Exactitud')
plt.xlabel('Epocas')
plt.legend(['Entrenamiento', 'Validacion'], loc='upper left')
plt.show()

In [ ]:
# Perdidas del modelo — misma grafica que en clase
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'], 'r--')
plt.title('Perdidas del modelo')
plt.ylabel('Perdidas')
plt.xlabel('Epocas')
plt.legend(['Entrenamiento', 'Validacion'], loc='upper left')
plt.show()

## 7. Matriz de Confusión (Seaborn)

Herramienta clave para identificar solapamientos entre clases similares (ej. shirt vs t-shirt).

In [ ]:
# Predicciones sobre el conjunto de test
test_data.reset()
predicciones = modelo.predict(test_data, verbose=1)
y_pred = np.argmax(predicciones, axis=1)
y_true = test_data.classes

print(f'Total predicciones: {len(y_pred)}')
print(f'Accuracy en test: {np.mean(y_pred == y_true):.4f}')

In [ ]:
# Matriz de Confusion con Seaborn (igual que en brief.md)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASES,
    yticklabels=CLASES
)
plt.title('Matriz de Confusion — RapidRelief AI (MobileNetV2)', fontsize=13)
plt.ylabel('Etiqueta real')
plt.xlabel('Etiqueta predicha')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Reporte completo por clase (precision, recall, F1)
print('Reporte de clasificacion por clase:')
print(classification_report(y_true, y_pred, target_names=CLASES))

## 8. Inferencia con imágenes individuales

Misma estructura que en clase: `load_img` → `img_to_array` → `expand_dims` → `preprocess_input` → `modelo.predict`.

In [ ]:
from tensorflow.keras.preprocessing import image

def predecir_prenda(img_path):
    img = image.load_img(img_path, target_size=IMAGE_SIZE)
    X = image.img_to_array(img)
    X = np.expand_dims(X, axis=0)
    X = preprocess_input(X)
    preds = modelo.predict(X, verbose=0)
    clase_idx = np.argmax(preds, axis=1)[0]
    clase_nombre = CLASES[clase_idx]
    confianza = preds[0][clase_idx]

    print(f'Prediccion: {clase_nombre} (confianza: {confianza:.2%})')
    print(f'Vector de probabilidades: {preds[0].round(3)}')

    plt.imshow(img)
    plt.title(f'Prediccion: {clase_nombre} ({confianza:.1%})')
    plt.axis('off')
    plt.show()
    return clase_nombre

In [ ]:
# Probar con una imagen del conjunto de test
# Modificar img_path con cualquier imagen real de tus carpetas de test
img_path = CLOTH_TEST + '/dress/' + os.listdir(CLOTH_TEST + '/dress/')[0]
predecir_prenda(img_path)

In [ ]:
# Probar con imagen de Fashion-MNIST convertida
from PIL import Image as PILImage
import pandas as pd

df_fmnist = pd.read_csv(FMNIST_TRAIN)
FMNIST_A_PROYECTO = {
    0: 't-shirt', 1: 'pants', 2: 'longsleeve', 3: 'dress',
    4: 'outwear', 5: 'shoes', 6: 'shirt', 7: 'shoes', 9: 'shoes'
}

for clase_fmnist, nombre_clase in [(3, 'dress'), (6, 'shirt'), (1, 'pants')]:
    fila = df_fmnist[df_fmnist['label'] == clase_fmnist].iloc[0]
    pixels = fila.drop('label').values.reshape(28, 28).astype('uint8')
    img_rgb = np.stack([pixels, pixels, pixels], axis=-1)
    img_pil = PILImage.fromarray(img_rgb, 'RGB').resize(IMAGE_SIZE, PILImage.BICUBIC)

    X = np.expand_dims(np.array(img_pil).astype('float32'), axis=0)
    X = preprocess_input(X)
    preds = modelo.predict(X, verbose=0)
    pred_idx = np.argmax(preds)

    print(f'FMNIST clase {clase_fmnist} ({nombre_clase}) -> Prediccion: {CLASES[pred_idx]} ({preds[0][pred_idx]:.2%})')

## 9. Exportar a TF Lite (para app móvil)

In [ ]:
import tensorflow as tf

# Convertir a TF Lite
converter = tf.lite.TFLiteConverter.from_keras_model(modelo)
tflite_model = converter.convert()

tflite_path = MODELS_DIR + '/rapidrelief_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f'Modelo TF Lite guardado: {tflite_path}')
print(f'Tamano: {size_mb:.1f} MB')

In [ ]:
# Guardar archivo de etiquetas para la app
labels_path = MODELS_DIR + '/labels.txt'
with open(labels_path, 'w') as f:
    for clase in CLASES:
        f.write(clase + '\n')

print('Archivo de etiquetas guardado:')
for i, clase in enumerate(CLASES):
    print(f'  {i}: {clase}')

## 10. Resumen final

In [ ]:
val_acc_final = max(history.history['val_accuracy'])
train_acc_final = max(history.history['accuracy'])

print('=== RESUMEN FINAL ===')
print(f'Modelo base:       MobileNetV2 (include_top=False, weights=imagenet)')
print(f'Cabeza:            Flatten -> Dense(2048,relu) -> Dense(512,relu) -> Dense(16,sigmoid) -> Dense(10,softmax)')
print(f'Optimizador:       SGD')
print(f'Loss:              categorical_crossentropy')
print(f'Train accuracy:    {train_acc_final:.4f}')
print(f'Val accuracy max:  {val_acc_final:.4f}')
print(f'KPI (>=0.90):      {"CUMPLIDO" if val_acc_final >= 0.90 else "NO cumplido — ver plan.md para siguiente paso"}')
print(f'Modelo H5:         {MODELS_DIR}/rapidrelief_mobilenetv2_v1.h5')
print(f'Modelo TFLite:     {MODELS_DIR}/rapidrelief_model.tflite ({size_mb:.1f} MB)')
print(f'Labels:            {MODELS_DIR}/labels.txt')